# Detecting noisy monitors

This notebook shows how to detect noisy monitors in a dataset using the WhyLabs Monitor Diagnoser. It uses the diagnoser to automatically detect the noisiest monitor for dataset, get a diagnosis of
the conditions causing the noise, get recommended changes and where automatable, apply those changes.

## Install requirements

In [31]:
# %pip install whylabs-toolkit[diagnoser]

## Setup whylabs API connection

First, set up the information to connect to WhyLabs. Update the org_id, dataset_id and api_key in the following before running it.


In [32]:
import getpass
from whylabs_toolkit.monitor.diagnoser.helpers.utils import env_setup

org_id = 'org-0'
dataset_id = 'model-0'
api_key = getpass.getpass()
api_endpoint = 'https://songbird.development.whylabsdev.com'

env_setup(
    org_id=org_id,
    dataset_id=dataset_id,
    api_key=api_key,
    whylabs_endpoint=api_endpoint
)

Initialize the Monitor Diagnoser with the org_id and dataset_id.

In [33]:
from whylabs_toolkit.monitor.diagnoser.monitor_diagnoser import MonitorDiagnoser
diagnoser = MonitorDiagnoser(org_id, dataset_id)

## Run the default diagnosis

With no further input, the diagnoser will make a series of calls to identify the noisiest monitor, segment and columns; and then perform a diagnosis.

In [34]:
monitor_report = diagnoser.diagnose()
monitor_report

MonitorDiagnosisReport(orgId='org-0', datasetId='model-0', analyzerId='kind-cyan-kangaroo-1253-analyzer', interval='2024-03-26T00:00:00.000Z/2024-04-25T00:00:00.000Z', expectedBatchCount=30, diagnosticData=DiagnosticDataSummary(diagnosticSegment=Segment(tags=[SegmentTag(key='purpose', value='car'), SegmentTag(key='verification_status', value='Source Verified')]), diagnosticProfile=ProfileSummary(minRowName='pred_credit_risk (output)', minRowCount=10473, maxRowName='pred_credit_risk (output)', maxRowCount=10473), diagnosticBatches=BatchesSummary(minBatchName='pred_credit_risk (output)', minBatchCount=30, maxBatchName='pred_credit_risk (output)', maxBatchCount=30), analysisResults=AnalysisResultsSummary(results=ResultRecord(diagnosedColumnCount=1, batchCount=30), failures=FailureRecord(totalFailuresCount=0, maxFailuresCount=0, meanFailuresCount=0, byColumnCount=[], byTypeCount=[]), anomalies=AnomalyRecord(totalAnomalyCount=30, maxAnomalyCount=30, meanAnomalyCount=30, batchCount=30, byCol

In [35]:
print(monitor_report.describe())

Diagnosis is for monitor "kind-cyan-kangaroo-1253" [kind-cyan-kangaroo-1253] in model-0 org-0, over interval 2024-03-26T00:00:00.000Z/2024-04-25T00:00:00.000Z.

Analyzer is drift configuration for histogram metric with TrailingWindow baseline.
Analyzer "kind-cyan-kangaroo-1253-analyzer" targets 1 columns and ran on 1 columns in the diagnosed segment.


Diagnostic segment is "purpose=car&verification_status=Source Verified".
Diagnostic interval contains 30 batches.

Diagnostic interval rollup contains 10473 rows for the diagnosed columns.

Analysis results summary:
Found non-failed results for 1 columns and 30 batches.
Found 30 anomalies in 1 columns, with up to 100.0% (30) batches having anomalies per column and 100.0% (30.0) on average.
Columns with anomalies are:
|    | 0                                 |
|---:|:----------------------------------|
|  0 | ('pred_credit_risk (output)', 30) |

No failures were detected.

No issues impacting diagnosis quality were detected
Conditions tha

The monitor report can be serialized to a JSON file for later use.

In [36]:
with open('monitor_report.json', 'w') as f:
    f.write(monitor_report.json())

In [37]:
from whylabs_toolkit.monitor.diagnoser.models import MonitorDiagnosisReport

with open('monitor_report.json', 'r') as f:
    monitor_report = MonitorDiagnosisReport.parse_raw(f.read())
print(monitor_report.json(indent=2))

{
  "orgId": "org-0",
  "datasetId": "model-0",
  "analyzerId": "kind-cyan-kangaroo-1253-analyzer",
  "interval": "2024-03-26T00:00:00.000Z/2024-04-25T00:00:00.000Z",
  "expectedBatchCount": 30,
  "diagnosticData": {
    "diagnosticSegment": {
      "tags": [
        {
          "key": "purpose",
          "value": "car"
        },
        {
          "key": "verification_status",
          "value": "Source Verified"
        }
      ]
    },
    "diagnosticProfile": {
      "minRowName": "pred_credit_risk (output)",
      "minRowCount": 10473,
      "maxRowName": "pred_credit_risk (output)",
      "maxRowCount": 10473
    },
    "diagnosticBatches": {
      "minBatchName": "pred_credit_risk (output)",
      "minBatchCount": 30,
      "maxBatchName": "pred_credit_risk (output)",
      "maxBatchCount": 30
    },
    "analysisResults": {
      "results": {
        "diagnosedColumnCount": 1,
        "batchCount": 30
      },
      "failures": {
        "totalFailuresCount": 0,
        "max

## Ask for recommended changes

Given the diagnosis report for the monitor, the ChangeRecommender will recommend changes to make to the monitor. By default it will make recommendations for all columns where it has detected noise-related conditions. Set the `min_anomaly_count` property to restrict this to only columns that caused a certain number of anomalies.


In [38]:
from whylabs_toolkit.monitor.diagnoser.recommendation.change_recommender import ChangeRecommender

recommender = ChangeRecommender(monitor_report)
recommender.min_anomaly_count = 1
changes = recommender.recommend()
print('\n'.join([f'{i+1}. {c.describe()}' for i, c in enumerate(changes)]))

1. Make a manual change to the analyzer to address small_nonnull_batches: less than 500 non-null records in 50% or more of the batches for ['pred_credit_risk (output)']


## Execute automatable changes

A subset of recommended changes can be executed automatically by the recommender. Pass the ones you want to make into the `make_changes` call, or pass all changes if you want it to make all of the automatable changes.

In [39]:
automatable_changes = [c for c in changes if c.can_automate()]
print('\n'.join([c.describe() for c in automatable_changes]))

In [40]:
change_results = recommender.make_changes(automatable_changes)
print(change_results.describe())

Note that the monitor will still appear to the diagnoser as the noisiest monitor until enough time has passed for the impact of the monitor changes to be observed. You may want to use the WhyLabs preview UI to view what impacts may be expected from the change.

## Reviewing other noisy monitors

The diagnoser can be used to review other noisy monitors in the dataset. The `noisy_monitors` property will return a list of the noisiest monitors, and the `monitor_id_to_diagnose` property can be set to the monitor_id of the monitor to diagnose.

In [41]:
import pandas as pd
noisy_monitors_df = pd.DataFrame.from_records([m.dict() for m in diagnoser.noisy_monitors])
noisy_monitors_df

,monitor_id,analyzer_id,metric,column_count,segment_count,anomaly_count,max_anomaly_per_column,min_anomaly_per_column,avg_anomaly_per_column,action_count,action_targets
0,kind-cyan-kangaroo-1253,kind-cyan-kangaroo-1253-analyzer,histogram,1,1,30,30,30,30,0,[]
1,cooperative-maroon-parrot-8886,discrete-drift-jensenshannon-analyzer,frequent_items,1,1,30,30,30,30,0,[]
2,famous-salmon-cobra-8902,famous-salmon-cobra-8902-analyzer,min,1,1,30,30,30,30,0,[]
3,proud-seagreen-carabeef-65,proud-seagreen-carabeef-65-analyzer,histogram,1,1,30,30,30,30,0,[]
4,None,cooperative-maroon-parrot-8886-analyzer,frequent_items,1,1,30,30,30,30,0,[]
...,...,...,...,...,...,...,...,...,...,...,...
94,glamorous-orchid-turtle-6425,glamorous-orchid-turtle-6425-analyzer,histogram,1,1,2,2,2,2,0,[]
95,breakable-limegreen-shrew-7623,breakable-limegreen-shrew-7623-analyzer,histogram,1,1,2,2,2,2,0,[]
96,hilarious-powderblue-chamois-8115,hilarious-powderblue-chamois-8115-analyzer,histogram,1,1,2,2,2,2,0,[]
97,horrible-magenta-sandpiper-8117,horrible-magenta-sandpiper-8117-analyzer,frequent_items,1,1,2,2,2,2,0,[]


In [42]:
diagnoser.monitor_id_to_diagnose = noisy_monitors_df.iloc[1]['monitor_id']
monitor_report = diagnoser.diagnose()
print(monitor_report.describe())

Diagnosis is for monitor "discrete-drift-jensenshannon" [cooperative-maroon-parrot-8886] in model-0 org-0, over interval 2024-03-26T00:00:00.000Z/2024-04-25T00:00:00.000Z.

Analyzer is drift configuration for frequent_items metric with TrailingWindow baseline.
Analyzer "discrete-drift-jensenshannon-analyzer" targets 30 columns and ran on 26 columns in the diagnosed segment.


Diagnostic segment is "overall".
Diagnostic interval contains 30 batches.

Diagnostic interval rollup contains 1945487 rows for the diagnosed columns.

Analysis results summary:
Found non-failed results for 26 columns and 30 batches.
Found 30 anomalies in 1 columns, with up to 100.0% (30) batches having anomalies per column and 100.0% (30.0) on average.
Columns with anomalies are:
|    | 0               |
|---:|:----------------|
|  0 | ('issue_d', 30) |

No failures were detected.

No issues impacting diagnosis quality were detected
Conditions that may contribute to noise include:
	* Condition changing_discrete (

You can also use the `noisy_monitors_with_actions` property to prioritize noise in monitors with actions, as these are most likely to cause alert fatigue.

In [43]:
pd.DataFrame.from_records([m.dict() for m in diagnoser.noisy_monitors_with_actions])


,monitor_id,analyzer_id,metric,column_count,segment_count,anomaly_count,max_anomaly_per_column,min_anomaly_per_column,avg_anomaly_per_column,action_count,action_targets
0,frequent-items-drift-monitor-u31vmb,frequent-items-drift-analyzer-u31vmb,frequent_items,2,1,31,30,1,15,2,"[email, slack]"
1,frequent-items-drift-monitor-uu0ax8,frequent-items-drift-analyzer-uu0ax8,frequent_items,2,1,31,30,1,15,3,"[email, slack, email-victor-at-whylabs]"
2,frequent-items-drift-monitor-48ukw1,frequent-items-drift-analyzer-48ukw1,frequent_items,2,1,31,30,1,15,2,"[email, slack]"
3,frequent-items-drift-monitor-jepz7t,frequent-items-drift-analyzer-jepz7t,frequent_items,2,1,31,30,1,15,2,"[email, slack]"
4,frequent-items-drift-monitor-pxexvn,frequent-items-drift-analyzer-pxexvn,frequent_items,2,1,31,30,1,15,2,"[email, slack]"
5,nice-burlywood-tarsier-4771,nice-burlywood-tarsier-4771-analyzer,unique_est,7,1,106,30,2,15,2,"[slack, email]"
6,energetic-black-cobra-7838,energetic-black-cobra-7838-analyzer,unique_est,7,1,80,30,2,11,1,[email]
7,elated-gray-baboon-4620,elated-gray-baboon-4620-analyzer,count_null_ratio,13,1,64,30,1,4,1,[email]
8,old-crimson-starling-2516,old-crimson-starling-2516-analyzer,frequent_items,2,1,24,23,1,12,1,[email]
9,uninterested-blueviolet-reindeer-9950,uninterested-blueviolet-reindeer-9950-analyzer,count,77,1,152,9,1,1,1,[christine-test-email]
